# SDXL Base — automated 48-image batch generatorPaste the complete `image-prompts.md` document from Telegram. This notebook accepts `Scene 01` followed by its prompt, generates **one image at a time automatically** for all 48 scenes, saves each image immediately, and packages the results into one ZIP.**Memory-conscious setup:** SDXL Base only (no Refiner), FP16, attention/VAE slicing, and model CPU offload. This is designed to reduce RAM/VRAM pressure on Colab Free, but a free runtime is not guaranteed to complete 48 images in one session. The notebook can resume completed scenes if the runtime's temporary files survive.**Run order:** install dependencies → restart runtime → run setup → paste prompts → validate → load model → generate batch → download ZIP.

In [ ]:
# Clean up mixed Pillow files left by previous Colab package upgrades, then install a tested-compatible version set.
import os, sys, glob, shutil, subprocess, site
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'Pillow'], check=False)
for base in site.getsitepackages():
    for pattern in ('PIL', 'Pillow-*.dist-info', 'pillow-*.dist-info'):
        for path in glob.glob(os.path.join(base, pattern)):
            if os.path.isdir(path): shutil.rmtree(path, ignore_errors=True)
            elif os.path.exists(path): os.remove(path)
packages = [
    'pillow==11.3.0',
    'transformers==4.48.2',
    'diffusers==0.30.3',
    'huggingface-hub>=0.25,<1.0',
    'accelerate>=0.30,<2.0',
    'safetensors',
    'ipywidgets>=7.7,<9',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall'] + packages)
print('Packages installed. NOW choose Runtime → Restart session before running any other cell. This restart is required to clear already-loaded Pillow modules.')

In [ ]:
import torch, os, re, io, zipfile, shutil, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown
import ipywidgets as widgets

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU (if available), then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
WORK = Path('/content/science-channel-image-job')
IMAGE_DIR = WORK / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
print('Working folder:', WORK)

## 1. Paste all 48 prompts togetherCopy the complete Telegram `image-prompts.md` document and paste it into the box. The accepted format includes `Scene 01` on one line and its prompt on the next line. You do not need to submit scenes separately.

In [ ]:
# This cell is self-contained: it imports the widget library even if earlier cells were skipped.
import ipywidgets as widgets
from google.colab import output
from IPython.display import display, Markdown

output.enable_custom_widget_manager()
prompt_box = widgets.Textarea(
    value='',
    placeholder='Paste the complete image-prompts.md text here…',
    description='Prompts:',
    layout=widgets.Layout(width='100%', height='360px'),
    style={'description_width': 'initial'}
)
display(prompt_box)
print('Paste all 48 scene prompts into the box above. Do not paste them into a code cell.')

In [ ]:
def parse_scene_prompts(markdown_text):
    # Accept both Markdown headings and the planner's plain-text format, including
    # 'Scene 01 prompt text...' where the prompt starts on the SAME line.
    heading_re = re.compile(r'(?mi)^\s*(?:#{1,6}\s*)?Scene\s*0*(\d+)\b[ \t]*(?::[ \t]*|-[ \t]*)?(.*)$')
    matches = list(heading_re.finditer(markdown_text))
    prompts = {}
    for index, match in enumerate(matches):
        number = int(match.group(1))
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown_text)
        # Keep any prompt text that appears after 'Scene 01' on the same line.
        inline_text = match.group(2).strip()
        following_text = markdown_text[match.end():end].strip()
        body = '\n'.join(part for part in (inline_text, following_text) if part).strip()
        body = re.sub(r'\n{3,}', '\n\n', body)
        body = re.sub(r'^\s*(?:---+|\*\*\*)\s*$', '', body, flags=re.M).strip()
        if not body:
            raise ValueError(f'Scene {number:02d} has no prompt text.')
        if number in prompts:
            raise ValueError(f'Duplicate scene heading: {number:02d}')
        prompts[number] = body
    expected = set(range(1, 49))
    if set(prompts) != expected:
        missing = sorted(expected - set(prompts))
        extra = sorted(set(prompts) - expected)
        preview = '\n'.join(markdown_text.splitlines()[:12])
        raise ValueError(
            f'Could not find exactly 48 scene headings. Found {len(prompts)}. '
            f'Missing: {missing}; unexpected: {extra}. '
            'Accepted formats include: Scene 01 prompt text, ## Scene 1: Heading, or ## Scene 01. '
            f'First lines received:\n{preview}'
        )
    return [prompts[n] for n in range(1, 49)]

PROMPTS = parse_scene_prompts(prompt_box.value)
print(f'Validated {len(PROMPTS)} numbered prompts.')
print('Scene 01 preview:', PROMPTS[0][:350] + ('…' if len(PROMPTS[0]) > 350 else ''))

## 2. Load SDXL Base (memory-conscious)This uses the public Stable Diffusion XL Base 1.0 model only. The Refiner is intentionally disabled to reduce memory use and improve the chance of completing the batch on a free Colab GPU. Model loading can still require several GB of RAM and a suitable GPU.

In [ ]:
import torchfrom diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteSchedulerbase_id = 'stabilityai/stable-diffusion-xl-base-1.0'pipe = StableDiffusionXLPipeline.from_pretrained(    base_id,    torch_dtype=torch.float16,    variant='fp16',    use_safetensors=True,    low_cpu_mem_usage=True)pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)pipe.enable_vae_slicing()pipe.enable_attention_slicing()# Keep only the active model component on the GPU to reduce VRAM demand.pipe.enable_model_cpu_offload()print('SDXL Base loaded. Refiner is disabled for lower memory use.')print('GPU:', torch.cuda.get_device_name(0))print('Generation mode: one image at a time, automated 48-scene batch.')

## 3. Generate all 48 images automaticallyClick Run once. The notebook processes scenes 01–48 sequentially without asking for input between images. Each image is saved as soon as it finishes. If Colab disconnects but preserves the runtime files, rerun the prompt validation and generation cells; valid completed images are skipped and missing scenes resume.**Note:** A Colab Free session can still run out of system RAM or time out. CPU offload and Base-only generation reduce memory use but cannot guarantee a 48-image run in every free session.

In [ ]:
from tqdm.auto import tqdmimport gcnegative_prompt = 'text, letters, words, watermark, logo, signature, blurry, low quality, low detail, poorly composed, distorted anatomy, extra limbs, malformed hands, duplicate subjects, flat lighting, oversaturated, jpeg artifacts, plastic texture, muddy details'style_suffix = ('Premium cinematic 3D CGI science documentary still, ultra-detailed professional visual development, physically based materials, realistic global illumination, nuanced volumetric lighting, rich but natural color grading, crisp focal subject, layered atmospheric depth, coherent scene design, polished film-quality rendering, wide 16:9 composition, no text or labels.')for scene_number, prompt in enumerate(tqdm(PROMPTS, desc='Generating 48-image batch'), start=1):    target = IMAGE_DIR / f'scene-{scene_number:02d}.webp'    if target.exists() and target.stat().st_size > 1000:        print(f'Scene {scene_number:02d}/48 already saved — skipping.')        continue    full_prompt = prompt.strip() + '\\n\\n' + style_suffix    generator = torch.Generator(device='cpu').manual_seed(42000 + scene_number)    try:        with torch.inference_mode():            result = pipe(                prompt=full_prompt,                negative_prompt=negative_prompt,                width=1024,                height=576,                num_inference_steps=35,                guidance_scale=6.5,                num_images_per_prompt=1,                generator=generator            )        # Persist each completed image immediately, before starting the next scene.        result.images[0].convert('RGB').save(target, 'WEBP', quality=88, method=4)        print(f'Saved scene {scene_number:02d}/48 → {target.name}')        del result        gc.collect()        if torch.cuda.is_available():            torch.cuda.empty_cache()    except torch.cuda.OutOfMemoryError as exc:        gc.collect()        torch.cuda.empty_cache()        raise RuntimeError(            f'GPU ran out of memory on scene {scene_number:02d}. '            'Completed images remain saved. Try a fresh Colab runtime and rerun; '            'the generator will skip valid completed scenes.'        ) from excprint('Batch generation finished. Run the ZIP cell to package all 48 images.')

## 4. Validate and download the ZIPThe notebook checks for all 48 numbered WebP images, packages them into one ZIP, and downloads it. The repository importer accepts ZIP files up to 25 MiB; if necessary, images are recompressed automatically to fit.

In [ ]:
files = sorted(IMAGE_DIR.glob('scene-*.webp'))if len(files) != 48:    raise RuntimeError(f'Expected 48 images, found {len(files)}. Generate missing scenes before continuing.')for number, path in enumerate(files, start=1):    expected = f'scene-{number:02d}.webp'    if path.name != expected or path.stat().st_size < 1000:        raise RuntimeError(f'Invalid or missing image: {expected}')zip_path = Path('/content/science-channel-48-images.zip')def build_zip():    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as archive:        for path in files:            archive.write(path, arcname=path.name)build_zip()size_mb = zip_path.stat().st_size / (1024 * 1024)if size_mb > 25:    print(f'ZIP is {size_mb:.1f} MiB; recompressing WebP images to quality 78 to meet the importer limit…')    for path in files:        with Image.open(path) as image:            image.convert('RGB').save(path, 'WEBP', quality=72, method=6)    build_zip()    size_mb = zip_path.stat().st_size / (1024 * 1024)print(f'Validated 48 images. ZIP size: {size_mb:.1f} MiB')if size_mb > 25:    raise RuntimeError('The ZIP is still larger than the repository importer limit of 25 MiB. Do not upload it; the images need further compression.')from google.colab import filesfiles.download(str(zip_path))

## 5. Send the ZIP to the existing render workflow
1. Download `science-channel-48-images.zip` above.
2. Open your GitHub repository on the `main` branch and upload the ZIP into `science-channel/telegram-uploads/` (create the folder if GitHub does not show it; normally it already exists).
3. Commit the ZIP to `main`.
4. Open **Actions → Telegram ZIP Import → Run workflow**.
5. For `episode_path`, copy the exact episode folder path from the caption of the Telegram prompt document. It looks like `science-channel/daily-episodes/YYYY-MM-DD/episode-001-topic-slug`.
6. For `zip_path`, enter `science-channel/telegram-uploads/science-channel-48-images.zip`.
7. Run the workflow. It validates/imports the images and the existing renderer takes over.

**Do not also send these images individually to the Telegram image receiver.** Use only one intake route for an episode.